# 04 — Evaluación del agente con preguntas fijas

13 preguntas fijas (`src/evaluacion_agente.py`). Se verifica **automáticamente**:
- **herramienta_ok**: llamó a todas las herramientas esperadas.
- **cita_ok**: la respuesta contiene la cita esperada (`[XM]`, `[Simulación MDP]`, `[Resolución …]`).

Las cifras de las respuestas se contrastaron manualmente con los datos originales (sección 3).

> Consume unas 30 llamadas al LLM (~3 min).

In [1]:
import sys, json
sys.path.insert(0, "../src")
import pandas as pd
pd.set_option("display.max_colwidth", 120)

In [2]:
from evaluacion_agente import evaluate, save
results = evaluate()
save(results)

[1/13] ¿Cuál fue el precio promedio de bolsa en marzo de 2026?


   tools=['consultar_precios'] tool_ok=True cita_ok=True


[2/13] ¿A qué hora del día fue más caro, en promedio, el precio de bolsa durante 2025?


   tools=['consultar_precios'] tool_ok=True cita_ok=True


[3/13] Compara el precio promedio de bolsa de enero de 2024 con el de enero de 2026.


   tools=['consultar_precios', 'consultar_precios'] tool_ok=True cita_ok=True


... (17 outputs left out)

   tools=[] tool_ok=False cita_ok=True


[13/13] ¿Cuál es la tarifa de energía de EPM para estrato 3 este mes?


   tools=[] tool_ok=True cita_ok=True


✅ Resultados en C:\Users\Tomas\OneDrive\Desktop\Archivos y programa EAFIT\Final IA\resultados


## 1. Resumen

In [3]:
print(f"Herramienta correcta: {results['herramienta_ok'].sum()}/{len(results)} ({results['herramienta_ok'].mean():.0%})")
print(f"Cita correcta:        {results['cita_ok'].sum()}/{len(results)} ({results['cita_ok'].mean():.0%})")
print(f"Iteraciones promedio: {results['iteraciones'].mean():.1f} | Tiempo promedio: {results['segundos'].mean():.1f} s")
results.groupby("tipo")[["herramienta_ok", "cita_ok"]].mean()

Herramienta correcta: 12/13 (92%)
Cita correcta:        13/13 (100%)
Iteraciones promedio: 1.8 | Tiempo promedio: 9.9 s


,herramienta_ok,cita_ok
tipo,,
RAG,1.0,1.0
caso límite,0.0,1.0
datos,1.0,1.0
explicación,1.0,1.0
fuera de alcance,1.0,1.0
multi-herramienta,1.0,1.0
simulación,1.0,1.0


In [4]:
results[["id", "tipo", "pregunta", "herramientas_esperadas", "herramientas_llamadas", "herramienta_ok", "cita_ok"]]

,id,tipo,pregunta,herramientas_esperadas,herramientas_llamadas,herramienta_ok,cita_ok
0,1,datos,¿Cuál fue el precio promedio de bolsa en marzo de 2026?,consultar_precios,consultar_precios,True,True
1,2,datos,"¿A qué hora del día fue más caro, en promedio, el precio de bolsa durante 2025?",consultar_precios,consultar_precios,True,True
2,3,datos,Compara el precio promedio de bolsa de enero de 2024 con el de enero de 2026.,consultar_precios,"consultar_precios, consultar_precios",True,True
3,4,simulación,¿Cuánto ahorra la política de Q-learning entre 2026-01-01 y 2026-10-31 con la batería por defecto?,simular,simular,True,True
4,5,simulación,¿Es mejor Q-learning que el horario fijo en el periodo de prueba de 2026? Justifica con cifras.,simular,"simular, simular",True,True
5,6,simulación,Simula iteración de valor de junio a agosto de 2026 con una batería de 20 kWh y 5 kW.,simular,simular,True,True
6,7,explicación,¿En qué horas carga y en qué horas descarga la política aprendida por Q-learning en 2026?,simular,simular,True,True
7,8,RAG,¿Qué es un SAEB según la regulación de la CREG?,buscar_documentos,buscar_documentos,True,True
8,9,RAG,"Según la resolución CREG 101 113 de 2026, ¿cómo se liquida la energía de los SAEB que son activos de red?",buscar_documentos,buscar_documentos,True,True
9,10,RAG,¿Qué es un autogenerador a pequeña escala según la CREG 174 de 2021?,buscar_documentos,buscar_documentos,True,True


## 2. Respuestas

In [5]:
for _, r in results.iterrows():
    print(f"\n[{r['id']}] {r['pregunta']}\n→ {r['respuesta'][:600]}")


[1] ¿Cuál fue el precio promedio de bolsa en marzo de 2026?
→ El precio promedio de bolsa en marzo de 2026 fue de **220,2 COP/kWh** [XM].

[2] ¿A qué hora del día fue más caro, en promedio, el precio de bolsa durante 2025?
→ Durante 2025, la hora del día con el precio de bolsa más caro en promedio fue las **19:00** (hora 19), con un promedio de **468,3 COP/kWh**. [XM]

[3] Compara el precio promedio de bolsa de enero de 2024 con el de enero de 2026.
→ Enero 2024 vs enero 2026 (precio de bolsa, COP/kWh):

- **Enero 2024:** promedio de **558,72 COP/kWh** (mínimo 95,85, máximo 721,16) [XM]
- **Enero 2026:** promedio de **212,39 COP/kWh** (mínimo 97,72, máximo 354,48) [XM]

El precio promedio bajó **346,33 COP/kWh** (aproximadamente un **62 %**) de enero 2024 a enero 2026 [XM].

[4] ¿Cuánto ahorra la política de Q-learning entre 2026-01-01 y 2026-10-31 con la batería por defecto?
→ La política de Q-learning ahorra **484.607 COP** entre 2026-01-01 y 2026-10-31 con la batería por defecto (1

## 3. Verificación de las cifras contra los datos originales

Si el agente usa la herramienta, los números deben coincidir con los calculados directamente.

In [6]:
import simulador
m = simulador.get_prices()
check = {
    "Promedio marzo 2026 (COP/kWh)": m.loc["2026-03"].to_numpy().mean(),
    "Promedio enero 2024": m.loc["2024-01"].to_numpy().mean(),
    "Promedio enero 2026": m.loc["2026-01"].to_numpy().mean(),
    "Hora más cara 2025": m.loc["2025"].mean().idxmax(),
}
table, _ = simulador.simulate_range("q_learning", "2026-01-01", "2026-10-31")
check["Ahorro Q-learning 2026 (COP)"] = table.loc["q_learning", "ahorro_total_cop"]
check["Ahorro horario fijo 2026 (COP)"] = table.loc["horario_fijo", "ahorro_total_cop"]
pd.Series(check).round(2)

Promedio marzo 2026 (COP/kWh)        220.20
Promedio enero 2024                  558.72
Promedio enero 2026                  212.39
Hora más cara 2025                    19.00
Ahorro Q-learning 2026 (COP)      484607.30
Ahorro horario fijo 2026 (COP)    469598.69
dtype: float64

## 4. Análisis y limitaciones observadas

- Las preguntas de datos, simulación y RAG usan la herramienta correcta y citan la fuente.
  Las cifras coinciden con los valores calculados directamente (sección 3).
- **Caso límite (precio en 2030):** el modelo suele responder **sin llamar** a `consultar_precios`, porque el
  system prompt ya le informa el rango de datos. La respuesta es correcta (no inventa un pronóstico),
  pero no sigue el camino esperado, que era llamar la herramienta y recibir el error. Por eso se marca como fallo de herramienta.
- **Fuera de alcance (tarifa EPM):** el agente reconoce que no tiene la información y no inventa cifras.
- **Regulación:** el agente distingue lo que dicen los fragmentos de lo que no. Por ejemplo, la Res. 101 113
  menciona que la política del MinEnergía faculta a la CREG para reglar el arbitraje, y el agente aclara qué
  no confirma el texto recuperado.
- **Limitaciones:** (1) el LLM no es determinista, así que los resultados pueden variar entre ejecuciones; (2) la
  verificación de la cita es sintáctica (busca el formato) y no comprueba que el fragmento respalde la
  afirmación; (3) el corpus solo tiene 4 resoluciones, así que normas no incluidas no se pueden responder.